# P2.8 — Simulasi Infiltrasi Tanah: Metode Horton
### Studio Pemrograman Python | Teknik Sipil

---

**Skenario:**  
Kota Semarang baru saja dilanda hujan lebat selama 8 jam. Sebagai analis hidrologi,
Anda diminta menghitung seberapa banyak air yang **meresap ke tanah** (infiltrasi)
vs yang **mengalir sebagai limpasan** (runoff) yang berpotensi menyebabkan banjir.

Gunakan **Metode Horton** untuk memodelkan laju infiltrasi terhadap waktu.

---

## Teori: Persamaan Horton

Laju infiltrasi menurun secara eksponensial seiring waktu:

$$f(t) = f_c + (f_0 - f_c) \cdot e^{-kt}$$

| Simbol | Keterangan | Satuan |
|--------|-----------|--------|
| $f(t)$ | Laju infiltrasi pada waktu $t$ | mm/jam |
| $f_0$  | Laju infiltrasi **awal** (tanah kering) | mm/jam |
| $f_c$  | Laju infiltrasi **akhir** (tanah jenuh) | mm/jam |
| $k$    | Konstanta deklinasi | jam⁻¹ |
| $t$    | Waktu | jam |

Infiltrasi **kumulatif** $F(t)$ adalah jumlah total air yang telah meresap:
$$F(t) = \int_0^t f(\tau)\,d\tau \approx \sum_{i} f(t_i) \cdot \Delta t$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

plt.style.use('seaborn-v0_8-whitegrid')
print('Library siap digunakan!')

---
## Step 1: Tentukan Parameter Tanah

Parameter di bawah merepresentasikan tanah **lempung berpasir (sandy loam)**.
Coba ubah nilainya dan amati perbedaan kurva yang dihasilkan!

In [ ]:
# ── Parameter Tanah ───────────────────────────────────────────────────────────
f0      = 50.0   # mm/jam  — laju infiltrasi awal (tanah kering)
fc      = 10.0   # mm/jam  — laju infiltrasi akhir (tanah jenuh)
k       = 2.0    # 1/jam   — konstanta deklinasi (lebih besar = lebih cepat turun)

# ── Pengaturan Waktu ──────────────────────────────────────────────────────────
dt      = 0.25   # jam     — interval waktu (15 menit)
t_total = 8.0    # jam     — durasi simulasi

# Buat array waktu dari 0 s.d. t_total dengan interval dt
t = np.arange(0, t_total + dt, dt)

print(f'Jumlah titik waktu : {len(t)}')
print(f'Waktu awal         : {t[0]} jam')
print(f'Waktu akhir        : {t[-1]} jam')

---
## Step 2: Hitung Laju Infiltrasi f(t)

Terapkan persamaan Horton untuk setiap titik waktu dalam array `t`.

In [ ]:
# TODO: Hitung laju infiltrasi f(t) menggunakan persamaan Horton
# Petunjuk: gunakan np.exp() untuk fungsi eksponensial
# Contoh penggunaan: np.exp(-2.0)  -->  e^(-2) ≈ 0.135

f = fc + (f0 - fc) * np.exp(-k * t)   # <-- lengkapi jika belum benar

# Verifikasi: nilai f harus dimulai mendekati f0 dan berakhir mendekati fc
print(f'f(t=0)   = {f[0]:.2f} mm/jam  (harusnya mendekati f0 = {f0})')
print(f'f(t=8)   = {f[-1]:.2f} mm/jam  (harusnya mendekati fc = {fc})')

---
## Step 3: Hitung Infiltrasi Kumulatif F(t)

Integrasikan $f(t)$ secara numerik untuk mendapatkan total air yang meresap.

In [ ]:
# TODO: Hitung infiltrasi kumulatif F(t)
# Metode numerik: F[i] = F[i-1] + f[i] * dt  (jumlah bertahap)
# Atau pakai np.cumsum() — penjumlahan kumulatif

F = np.cumsum(f * dt)   # <-- kamu bisa juga coba cara loop di bawah

# Cara alternatif dengan loop (hapus tanda # untuk mencoba):
# F = np.zeros(len(t))
# for i in range(1, len(t)):
#     F[i] = F[i-1] + f[i] * dt

print(f'Total infiltrasi setelah {t_total} jam : {F[-1]:.1f} mm')
print(f'Rata-rata laju infiltrasi           : {F[-1]/t_total:.1f} mm/jam')

---
## Step 4: Tampilkan Tabel Hasil

In [ ]:
# Tampilkan setiap 1 jam (bukan setiap 0.25 jam agar tabel tidak terlalu panjang)
step = int(1.0 / dt)   # berapa baris per jam

tabel = pd.DataFrame({
    'Waktu (jam)'                  : t[::step],
    'Laju Infiltrasi f (mm/jam)'   : f[::step].round(2),
    'Infiltrasi Kumulatif F (mm)'  : F[::step].round(2)
})
tabel.set_index('Waktu (jam)', inplace=True)
print(tabel.to_string())

---
## Step 5: Visualisasi Kurva Infiltrasi

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# ── Kiri: Laju infiltrasi f(t) ───────────────────────────────────────────────
ax1.plot(t, f, 'steelblue', linewidth=2.5, label='Laju infiltrasi f(t)')
ax1.axhline(fc, color='tomato', linestyle='--', linewidth=1.5,
            label=f'Laju akhir $f_c$ = {fc} mm/jam')
ax1.fill_between(t, fc, f, alpha=0.15, color='steelblue')
ax1.set_xlabel('Waktu (jam)', fontsize=12)
ax1.set_ylabel('Laju Infiltrasi (mm/jam)', fontsize=12)
ax1.set_title('Kurva Laju Infiltrasi Horton', fontsize=12, fontweight='bold')
ax1.legend(fontsize=10)

# ── Kanan: Infiltrasi kumulatif F(t) ─────────────────────────────────────────
ax2.fill_between(t, F, alpha=0.2, color='seagreen')
ax2.plot(t, F, 'seagreen', linewidth=2.5, label='Infiltrasi kumulatif F(t)')
ax2.set_xlabel('Waktu (jam)', fontsize=12)
ax2.set_ylabel('Infiltrasi Kumulatif (mm)', fontsize=12)
ax2.set_title('Infiltrasi Kumulatif', fontsize=12, fontweight='bold')
ax2.legend(fontsize=10)

plt.suptitle(f'Simulasi Infiltrasi — Metode Horton\n'
             f'$f_0$={f0}, $f_c$={fc}, k={k} jam⁻¹',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('P2.8_infiltrasi.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafik disimpan sebagai P2.8_infiltrasi.png')

---
## Level 2 — Bandingkan Berbagai Jenis Tanah

Jalankan sel di bawah untuk membandingkan laju infiltrasi **pasir, lempung, dan gambut**.
Perhatikan perbedaan kecepatan penurunannya — ini yang menentukan seberapa besar banjir yang terjadi!

In [ ]:
soil_types = {
    'Pasir (Sandy)'       : {'f0': 120, 'fc': 30, 'k': 3.5, 'color': 'orange',    'ls': '-'},
    'Lempung Berpasir'    : {'f0':  50, 'fc': 10, 'k': 2.0, 'color': 'steelblue', 'ls': '-'},
    'Lempung (Clay)'      : {'f0':  20, 'fc':  3, 'k': 0.8, 'color': 'brown',     'ls': '-'},
    'Gambut (Peat)'       : {'f0': 180, 'fc': 60, 'k': 4.5, 'color': 'seagreen',  'ls': '--'},
}

fig, ax = plt.subplots(figsize=(10, 5))

for soil, p in soil_types.items():
    f_s = p['fc'] + (p['f0'] - p['fc']) * np.exp(-p['k'] * t)
    ax.plot(t, f_s, color=p['color'], linestyle=p['ls'], linewidth=2.2, label=soil)

ax.set_xlabel('Waktu (jam)', fontsize=12)
ax.set_ylabel('Laju Infiltrasi (mm/jam)', fontsize=12)
ax.set_title('Perbandingan Laju Infiltrasi Berbagai Jenis Tanah', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

# TODO — Pertanyaan Diskusi:
# 1. Tanah jenis apa yang paling cepat jenuh (f mendekati fc)?
# 2. Pada kondisi hujan 30 mm/jam, tanah mana yang menghasilkan runoff terbesar?
# 3. Apa implikasinya bagi perencanaan drainase kota?

---
## Level 3 — Hitung Runoff dari Data Hujan

Jika intensitas hujan $i(t)$ diketahui, runoff terjadi saat $i(t) > f(t)$.

$$\text{Runoff}(t) = \max(0,\; i(t) - f(t))$$

Sel di bawah menggunakan hujan sintetik — coba ganti dengan data riil dari file CSV!

In [ ]:
# Hujan sintetik: intensitas konstan 25 mm/jam selama 8 jam
rain_intensity = np.full_like(t, fill_value=25.0)   # mm/jam

# TODO: Hitung runoff = bagian hujan yang tidak bisa meresap
runoff = np.maximum(0, rain_intensity - f)

fig, ax = plt.subplots(figsize=(10, 5))
ax.fill_between(t, rain_intensity, label='Intensitas Hujan', color='steelblue', alpha=0.3)
ax.plot(t, rain_intensity, 'steelblue', linewidth=2)
ax.fill_between(t, f, label='Kapasitas Infiltrasi f(t)', color='seagreen', alpha=0.4)
ax.plot(t, f, 'seagreen', linewidth=2)
ax.fill_between(t, f, rain_intensity, where=(rain_intensity > f),
                color='tomato', alpha=0.5, label='Runoff (limpasan)')

ax.set_xlabel('Waktu (jam)', fontsize=12)
ax.set_ylabel('Intensitas (mm/jam)', fontsize=12)
ax.set_title('Partisi Hujan: Infiltrasi vs Runoff', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

total_rain   = np.trapz(rain_intensity, t)
total_infilt = F[-1]
total_runoff = np.trapz(runoff, t)
print(f'Total hujan       : {total_rain:.1f} mm')
print(f'Total infiltrasi  : {total_infilt:.1f} mm')
print(f'Total runoff      : {total_runoff:.1f} mm  ({total_runoff/total_rain*100:.1f}% dari hujan)')